# CredLens Credit Portfolio Case Study

**All data in this notebook is 100% synthetic**, produced by CredLens's own data-generation process (DGP) - a portfolio project simulating a Brazilian digital credit fintech. Nothing here describes a real financial institution, a real customer, or a real portfolio. See `docs/assumptions_and_limitations.md`.

This notebook is a **thin, read-only viewer** over outputs an already-run `credlens analysis run` produced under `reports/portfolio_analysis/` - it loads CSV tables and PNG figures and narrates them; it does not recompute any KPI, run any SQL, or duplicate logic that lives in `src/credlens/analysis/`. For the full narrative, see:

- Executive summary: [`../reports/portfolio_analysis/executive_summary.md`](../reports/portfolio_analysis/executive_summary.md) (English) / [`executive_summary.pt-BR.md`](../reports/portfolio_analysis/executive_summary.pt-BR.md) (Portuguese)
- Technical report: [`../reports/portfolio_analysis/technical_report.md`](../reports/portfolio_analysis/technical_report.md) / [`technical_report.pt-BR.md`](../reports/portfolio_analysis/technical_report.pt-BR.md)
- Business-question registry: [`../analysis/questions.yml`](../analysis/questions.yml)

## Reproducing the data this notebook reads

```bash
uv sync --extra warehouse --extra analysis
uv run credlens warehouse build --suite-id SUITE_sample_2026
uv run credlens analysis run --build-id <build_id_from_the_previous_command>
```

Then restart this notebook's kernel and run all cells top to bottom - every cell below only reads files, it never re-runs a query or invokes the generator itself.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

REPORTS_DIR = Path("..") / "reports" / "portfolio_analysis"
TABLES_DIR = REPORTS_DIR / "tables"
FIGURES_DIR = REPORTS_DIR / "figures"
MANIFEST_PATH = REPORTS_DIR / "manifest.json"

if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"No analysis manifest found at '{MANIFEST_PATH}'. Run the reproduction "
        "commands in the cell above first (`credlens warehouse build` then "
        "`credlens analysis run`), then restart this notebook's kernel."
    )

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))


def show_table(name: str, max_rows: int = 10) -> None:
    df = pd.read_csv(TABLES_DIR / f"{name}.csv")
    display(df.head(max_rows))


def show_figure(name: str) -> None:
    path = FIGURES_DIR / f"{name}.png"
    if path.is_file():
        display(Image(filename=str(path)))
    else:
        display(Markdown(f"_Figure `{name}` was not generated in this analysis run._"))

In [ ]:
n_tables = len(manifest["tables_written"])
n_figures = len(manifest["figures_written"])
display(
    Markdown(
        f"**Build:** `{manifest['build_id']}`  \n"
        f"**Suite:** `{manifest.get('suite_id')}`  \n"
        f"**Analysis ID:** `{manifest['analysis_id']}`  \n"
        f"**Warehouse fingerprint:** `{manifest['warehouse_fingerprint'][:24]}...`  \n"
        f"**Final status:** `{manifest['final_status']}`  \n"
        f"**Tables / figures written:** {n_tables} / {n_figures}  \n"
        f"**Warnings:** {len(manifest['warnings']) or 'none'}"
    )
)

## 1. Credit Funnel

Submitted → decisioned → approved → booked applications, by scenario. See `analysis/questions.yml` FUN-01/FUN-02/FUN-03.

In [ ]:
show_table("funnel_monthly")
show_figure("credit_funnel")

## 2. Portfolio Composition and Outstanding Balance

Outstanding balance is a STOCK measure, read as of each `snapshot_date` - never summed across dates. See COMP-01/COMP-02/COMP-03.

In [ ]:
show_table("portfolio_monthly")
show_figure("outstanding_balance_over_time")

## 3. Delinquency (PAR30/60/90)

Balance-weighted portfolio-at-risk over time, baseline vs. scenarios. See DEL-01/DEL-02/DEL-03.

In [ ]:
show_table("delinquency_monthly")
show_figure("par_curves")
show_figure("roll_rate_heatmap")

## 4. Vintage Cohorts

90+ incidence by months-on-book (MOB), one line per origination cohort - cohorts are only ever compared up to the MOB **both** have actually reached. See VIN-01/VIN-02.

In [ ]:
show_table("vintage_cohorts")
show_figure("vintage_curves")

## 5. Cure and Relapse

Among contracts that ever cured from delinquency, how many later redefaulted. See CUR-01/CUR-02.

In [ ]:
show_table("cure_and_redefault")
show_figure("cure_and_relapse")

## 6. Write-off and Recovery

`recovery_rate` reflects this DGP's own configured recovery-probability/amount rule - not a real collections operation's performance (no LGD/EAD modeling exists in this project). See COL-01/COL-02/COL-03.

In [ ]:
show_table("writeoff_recovery")
show_figure("writeoff_and_recovery")
show_table("collections_performance")

## 7. Scenario Comparison

Baseline vs. `policy_expansion` / `policy_tightening` / `macroeconomic_stress` / `collections_change`, paired by common random numbers (CRN) within the same suite. `collections_change` varies only AGGREGATE, scenario-level DGP parameters - there is **no per-contact causal attribution** in these results. See SCN-01.

In [ ]:
show_table("scenario_comparison")
show_figure("policy_scenario_comparison")

## 8. Macroeconomic Stress: Pre- vs. Post-Shock

The pre-shock period must show a near-zero delta (baseline and `macroeconomic_stress` are indistinguishable before the shock date by DGP design, enforced by `warehouse/tests/assert_pre_shock_period_identical_across_scenarios.sql`); only the post-shock period should diverge. See SCN-02.

In [ ]:
show_table("macro_stress_pre_post")
show_figure("macro_stress_pre_post")

## 9. Segmentation

Every segmented breakdown below flags `low_sample=True` for cells below `MIN_SEGMENT_OBSERVATIONS=10` rather than dropping them - never read a `low_sample` cell as a stable rate. See `analysis/specifications/segmentation_policy.md`, FUN-02, COMP-02, COMP-03.

In [ ]:
show_table("funnel_by_channel_and_scenario")
show_table("portfolio_by_region_and_channel")
show_table("policy_version_comparison")

## 10. Quality and Provenance

Every number above traces back to a specific dbt-tested, independently-reconciled warehouse build - not a hand-typed figure. This scorecard summarizes that build's own quality signals.

In [ ]:
show_figure("quality_provenance_scorecard")
show_figure("public_benchmark_overview")
display(Markdown(f"Full manifest: `{MANIFEST_PATH}`"))

## 11. Limitations

- Every figure above is synthetic; there is no claim of real-institution representativeness.
- No revenue, cost, LGD, EAD, or regulatory PD data exists in this project - nothing here supports a profitability or capital-adequacy conclusion.
- Scenario comparisons are only valid **within the same `suite_id`** (same seed / common random numbers).
- `collections_change` results describe a synthetic counterfactual DGP configuration, not a real collections strategy's causal effect.
- This notebook shows one build/suite/seed. See the technical report's "Multi-seed robustness" section (when `credlens analysis run --multiseed` was used) for how much a single-seed delta moves across seeds - always labeled **simulation variability**, never a statistical confidence interval.
- Out of scope entirely: dashboards, trained predictive models, cutoff optimization, real causal inference, and any definitive credit-policy recommendation. See `docs/roadmap.md`.